# Pipeline Tiền Xử Lý Dữ Liệu NÂNG CẤP (Enhanced Preprocessing Pipeline)

Notebook này là phiên bản **nâng cấp** của pipeline tiền xử lý, bổ sung thêm nhiều đặc trưng mới:
- **Thời tiết**: 6 biến gốc + 11 biến phái sinh (severity, binary indicators, diffs)
- **Cyclical Encoding**: sin/cos cho MONTH, HOUR, DOW
- **Congestion nâng cao**: Mật độ tại sân bay đích, tổng chuyến bay hàng ngày
- **Rolling Window Delays**: Trung bình trễ 12 tháng gần nhất (thay vì tích lũy toàn bộ)
- **Target Encoding**: Tỷ lệ trễ lịch sử theo carrier/origin/route/dest

Tổng số features: **~50** (tăng từ 22 features của phiên bản trước).

### Kiến trúc vẫn giữ nguyên:
- Batching theo từng năm (Year-by-Year) để tiết kiệm RAM
- Temporal Split: Train (2016-2022), Valid (2023), Test (2024)
- Zero Data Leakage: Historical accumulators chỉ dùng dữ liệu quá khứ
- Loại trừ năm Covid 2020 khỏi accumulators

### Cell 1: Thiết lập môi trường & Kiểm soát bộ nhớ (Setup & Memory Management)

In [1]:
import os
import sys
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt

# Cấu hình cảnh báo và hiển thị pandas
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)

# Hàm kiểm tra mức tiêu thụ RAM hiện tại của tiến trình
def get_memory_usage_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)

# Hàm tự động định vị thư mục gốc của repository
def resolve_repo_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]
    for candidate in candidates:
        if (candidate / "src" / "data" / "processed").exists() or (candidate / "data" / "processed").exists():
            return candidate
    return Path.cwd()

repo_root = resolve_repo_root()
print(f"Repository Root: {repo_root}")
print(f"RAM ban đầu của tiến trình Python: {get_memory_usage_mb():.1f} MB")

Repository Root: d:\Documents\BaiTap\KhoaLuanCuNhan\aeolus-gate-optimization-1\src
RAM ban đầu của tiến trình Python: 130.6 MB


### Cell 2: Cấu hình Pipeline & Lựa chọn nguồn dữ liệu (Pipeline Configuration)
Bạn có thể lựa chọn:
* `DATA_SOURCE = "tabular_by_year"`: Chạy trên toàn bộ mạng bay nước Mỹ (**54,674,003 dòng**).
* `DATA_SOURCE = "inbound_atl"`: Chạy trên các chuyến bay đến Atlanta (**3,022,433 dòng**).
Pipeline batching tự động thích ứng với cả hai nguồn dữ liệu.

In [3]:
# CẤU HÌNH PIPELINE
DATA_SOURCE = "tabular_by_year"  # Tùy chọn: 'tabular_by_year' (54.6M dòng) hoặc 'inbound_atl' (3.02M dòng)
YEARS_TO_PROCESS = list(range(2016, 2025))  # Xử lý toàn bộ 9 năm từ 2016 đến 2024

candidate_paths = [
    repo_root / "src" / "data" / "processed" / DATA_SOURCE,
    repo_root / "data" / "processed" / DATA_SOURCE,
    Path(f"src/data/processed/{DATA_SOURCE}"),
    Path(f"../data/processed/{DATA_SOURCE}"),
    Path(f"../../src/data/processed/{DATA_SOURCE}"),
    Path(f"../../data/processed/{DATA_SOURCE}")
]

data_dir = None
for p in candidate_paths:
    if p.exists():
        data_dir = p
        break

if data_dir is None:
    raise FileNotFoundError(f"Không tìm thấy thư mục dữ liệu '{DATA_SOURCE}'! Vui lòng kiểm tra lại đường dẫn.")

# Danh sách 23 cột cần thiết (NÂNG CẤP: thêm 6 cột thời tiết so với bản gốc 17 cột)
PROJECTION_COLUMNS = [
    "source_year", "FL_DATE", "OP_CARRIER", "ORIGIN", "DEST",
    "CRS_DEP_TIME", "CRS_ARR_TIME", "CRS_ELAPSED_TIME",
    "DEP_DELAY", "ARR_DELAY",
    "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK",
    "O_LATITUDE", "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE",
    # --- MỚI: 6 cột thời tiết từ Meteostat ---
    "O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD"
]

print(f"Nguồn dữ liệu được chọn : {DATA_SOURCE}")
print(f"Đường dẫn thư mục       : {data_dir}")
print(f"Các năm sẽ được xử lý   : {YEARS_TO_PROCESS}")
print(f"Số lượng cột đọc từ đĩa : {len(PROJECTION_COLUMNS)} cột (bao gồm 6 cột thời tiết mới)")

Nguồn dữ liệu được chọn : tabular_by_year
Đường dẫn thư mục       : d:\Documents\BaiTap\KhoaLuanCuNhan\aeolus-gate-optimization-1\src\data\processed\tabular_by_year
Các năm sẽ được xử lý   : [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Số lượng cột đọc từ đĩa : 23 cột (bao gồm 6 cột thời tiết mới)


### Cell 3: Hàm Làm sạch & Kiểm toán toàn vẹn Dữ liệu theo Batch (Data Cleaning & Consistency)
Áp dụng các kiểm toán từ Notebook 02:
* Loại bỏ missing values cốt lõi.
* Kiểm toán thời gian bay âm/bằng 0 (`CRS_ELAPSED_TIME > 0`).
* Kiểm toán trễ ngoại lai phi lý (`-300 <= DELAY <= 2000`).
* Kiểm toán tọa độ địa lý hợp lệ.

In [4]:
def clean_and_filter_batch(df_batch):
    initial_count = len(df_batch)
    
    # 1. Loại bỏ các dòng khuyết thông tin cốt lõi
    core_cols = ["CRS_ELAPSED_TIME", "ARR_DELAY", "DEP_DELAY", "CRS_DEP_TIME", "CRS_ARR_TIME"]
    df_clean = df_batch.dropna(subset=core_cols).copy()
    
    # 2. Lọc bỏ các giá trị bất thường logic (Consistency Checks)
    valid_mask = (
        (df_clean["CRS_ELAPSED_TIME"] > 0) &
        (df_clean["ARR_DELAY"] >= -300) & (df_clean["ARR_DELAY"] <= 2000) &
        (df_clean["DEP_DELAY"] >= -300) & (df_clean["DEP_DELAY"] <= 2000) &
        (df_clean["O_LATITUDE"] >= -90) & (df_clean["O_LATITUDE"] <= 90) &
        (df_clean["O_LONGITUDE"] >= -180) & (df_clean["O_LONGITUDE"] <= 180) &
        (df_clean["D_LATITUDE"] >= -90) & (df_clean["D_LATITUDE"] <= 90) &
        (df_clean["D_LONGITUDE"] >= -180) & (df_clean["D_LONGITUDE"] <= 180)
    )
    df_clean = df_clean[valid_mask].copy()
    removed_count = initial_count - len(df_clean)
    
    return df_clean, removed_count

print("Hàm clean_and_filter_batch đã sẵn sàng!")

Hàm clean_and_filter_batch đã sẵn sàng!


### Cell 4: Hàm Kiến tạo Đặc trưng NÂNG CẤP (Enhanced Feature Engineering)
Bao gồm tất cả các đặc trưng gốc + các đặc trưng mới:
* **Thời tiết**: 6 biến gốc (nhiệt độ, lượng mưa, tốc độ gió) + 11 biến phái sinh
* **Cyclical Encoding**: Mã hóa tuần hoàn sin/cos cho thời gian
* **Congestion nâng cao**: Mật độ sân bay đích, tổng chuyến bay hàng ngày
* **Rolling Window Delays**: Trung bình trễ 12 tháng gần nhất
* **Target Encoding**: Tỷ lệ trễ lịch sử (delay rate)

In [5]:
def haversine_vectorized(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return (6367.0 * c).astype("float32")

def engineer_features_batch(df_batch, historical_lookups=None, rolling_lookups=None, delay_rate_lookups=None):
    """
    Tạo toàn bộ đặc trưng cho batch hiện tại (PHIÊN BẢN NÂNG CẤP).
    Bao gồm: schedule, weather, cyclical, congestion, historical delays, target encoding.
    """
    # ═══════════════════════════════════════════════════════════════
    # 1. Trích xuất giờ khởi hành và giờ đến dự kiến
    # ═══════════════════════════════════════════════════════════════
    df_batch["DEP_HOUR"] = df_batch["CRS_DEP_TIME"].str.slice(11, 13).astype("int8")
    df_batch["ARR_HOUR"] = df_batch["CRS_ARR_TIME"].str.slice(11, 13).astype("int8")
    
    # ═══════════════════════════════════════════════════════════════
    # 2. Đặc trưng lịch trình cơ bản
    # ═══════════════════════════════════════════════════════════════
    df_batch["YEAR"] = df_batch["source_year"].astype("int16")
    df_batch["MONTH"] = df_batch["MONTH"].astype("int8")
    df_batch["DAY"] = df_batch["DAY_OF_MONTH"].astype("int8")
    df_batch["DOW"] = df_batch["DAY_OF_WEEK"].astype("int8")
    df_batch["QUARTER"] = ((df_batch["MONTH"] - 1) // 3 + 1).astype("int8")
    df_batch["IS_WEEKEND"] = (df_batch["DOW"] >= 5).astype("int8")
    df_batch["PEAK_HOUR"] = (((df_batch["DEP_HOUR"] >= 6) & (df_batch["DEP_HOUR"] <= 9)) | ((df_batch["DEP_HOUR"] >= 16) & (df_batch["DEP_HOUR"] <= 19))).astype("int8")
    df_batch["IS_COVID_PERIOD"] = (df_batch["YEAR"] == 2020).astype("int8")
    
    # 3. Mùa và Buổi trong ngày
    season_map = {12: 0, 1: 0, 2: 0, 3: 1, 4: 1, 5: 1, 6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3}
    df_batch["SEASON_CODE"] = df_batch["MONTH"].map(season_map).astype("int8")
    df_batch["TIME_OF_DAY_CODE"] = pd.cut(df_batch["DEP_HOUR"], bins=[-1, 5, 11, 17, 24], labels=[0, 1, 2, 3]).astype("int8")
    
    # ═══════════════════════════════════════════════════════════════
    # 4. Cự ly bay & Phân nhóm
    # ═══════════════════════════════════════════════════════════════
    df_batch["DISTANCE_KM"] = haversine_vectorized(
        df_batch["O_LONGITUDE"], df_batch["O_LATITUDE"],
        df_batch["D_LONGITUDE"], df_batch["D_LATITUDE"]
    )
    df_batch["DISTANCE_GROUP_CODE"] = pd.cut(
        df_batch["DISTANCE_KM"],
        bins=[-np.inf, 500, 1000, 1500, 2500, np.inf],
        labels=[0, 1, 2, 3, 4]
    ).astype("int8")
    df_batch["LONG_HAUL"] = (df_batch["DISTANCE_KM"] >= 1500).astype("int8")
    
    # ═══════════════════════════════════════════════════════════════
    # 5. Tuyến bay & Mật độ nghẽn (GIỮ NGUYÊN + NÂNG CẤP)
    # ═══════════════════════════════════════════════════════════════
    df_batch["ROUTE"] = df_batch["ORIGIN"].astype(str) + "_" + df_batch["DEST"].astype(str)
    df_batch["ORIGIN_CONGESTION_SAME_HOUR"] = (
        df_batch.groupby(["FL_DATE", "ORIGIN", "DEP_HOUR"])["CRS_ELAPSED_TIME"]
        .transform("count")
        .astype("int16")
    )
    # MỚI: Mật độ đến cùng giờ tại sân bay đích
    df_batch["DEST_CONGESTION_SAME_HOUR"] = (
        df_batch.groupby(["FL_DATE", "DEST", "ARR_HOUR"])["CRS_ELAPSED_TIME"]
        .transform("count")
        .astype("int16")
    )
    # MỚI: Tổng chuyến bay xuất phát từ origin trong ngày
    df_batch["ORIGIN_DAILY_FLIGHTS"] = (
        df_batch.groupby(["FL_DATE", "ORIGIN"])["CRS_ELAPSED_TIME"]
        .transform("count")
        .astype("int16")
    )
    # MỚI: Tổng chuyến bay đến dest trong ngày
    df_batch["DEST_DAILY_FLIGHTS"] = (
        df_batch.groupby(["FL_DATE", "DEST"])["CRS_ELAPSED_TIME"]
        .transform("count")
        .astype("int16")
    )
    
    # ═══════════════════════════════════════════════════════════════
    # 6. MỚI: ĐẶC TRƯNG THỜI TIẾT (WEATHER FEATURES)
    # ═══════════════════════════════════════════════════════════════
    # a. Giữ nguyên 6 biến thời tiết gốc, xử lý missing bằng median
    for wcol in ["O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD"]:
        df_batch[wcol] = df_batch[wcol].fillna(df_batch[wcol].median()).astype("float32")
    
    # b. Các chỉ báo thời tiết xấu (Binary Weather Severity Indicators)
    df_batch["IS_ORIGIN_RAINY"] = (df_batch["O_PRCP"] > 0).astype("int8")
    df_batch["IS_DEST_RAINY"] = (df_batch["D_PRCP"] > 0).astype("int8")
    df_batch["IS_ORIGIN_WINDY"] = (df_batch["O_WSPD"] >= 25).astype("int8")  # >= 25 km/h
    df_batch["IS_DEST_WINDY"] = (df_batch["D_WSPD"] >= 25).astype("int8")
    df_batch["IS_ORIGIN_COLD"] = (df_batch["O_TEMP"] <= 0).astype("int8")  # Freezing conditions
    df_batch["IS_DEST_COLD"] = (df_batch["D_TEMP"] <= 0).astype("int8")
    
    # c. Chênh lệch thời tiết Origin vs Dest
    df_batch["TEMP_DIFF"] = (df_batch["O_TEMP"] - df_batch["D_TEMP"]).astype("float32")
    df_batch["WSPD_DIFF"] = (df_batch["O_WSPD"] - df_batch["D_WSPD"]).astype("float32")
    
    # d. Composite Weather Severity Score
    df_batch["WEATHER_SEVERITY_ORIGIN"] = (
        df_batch["IS_ORIGIN_RAINY"] + df_batch["IS_ORIGIN_WINDY"] + df_batch["IS_ORIGIN_COLD"]
    ).astype("int8")
    df_batch["WEATHER_SEVERITY_DEST"] = (
        df_batch["IS_DEST_RAINY"] + df_batch["IS_DEST_WINDY"] + df_batch["IS_DEST_COLD"]
    ).astype("int8")
    df_batch["WEATHER_SEVERITY_TOTAL"] = (
        df_batch["WEATHER_SEVERITY_ORIGIN"] + df_batch["WEATHER_SEVERITY_DEST"]
    ).astype("int8")
    
    # ═══════════════════════════════════════════════════════════════
    # 7. MỚI: CYCLICAL ENCODING (Mã hóa tuần hoàn)
    # ═══════════════════════════════════════════════════════════════
    df_batch["MONTH_SIN"] = np.sin(2 * np.pi * df_batch["MONTH"] / 12).astype("float32")
    df_batch["MONTH_COS"] = np.cos(2 * np.pi * df_batch["MONTH"] / 12).astype("float32")
    df_batch["HOUR_SIN"] = np.sin(2 * np.pi * df_batch["DEP_HOUR"] / 24).astype("float32")
    df_batch["HOUR_COS"] = np.cos(2 * np.pi * df_batch["DEP_HOUR"] / 24).astype("float32")
    df_batch["DOW_SIN"] = np.sin(2 * np.pi * df_batch["DOW"] / 7).astype("float32")
    df_batch["DOW_COS"] = np.cos(2 * np.pi * df_batch["DOW"] / 7).astype("float32")
    
    # ═══════════════════════════════════════════════════════════════
    # 8. Gán độ trễ lịch sử từ bộ tích lũy quá khứ (Cumulative)
    # ═══════════════════════════════════════════════════════════════
    if historical_lookups is not None:
        for col_name, lookup_dict, default_val in [
            ("ROUTE_AVG_ARR_DELAY_PAST",   historical_lookups.get("route_arr", {}),   0.0),
            ("AIRLINE_AVG_ARR_DELAY_PAST", historical_lookups.get("carrier_arr", {}), 0.0),
            ("ORIGIN_AVG_ARR_DELAY_PAST",  historical_lookups.get("origin_arr", {}),  0.0),
            ("ROUTE_AVG_DEP_DELAY_PAST",   historical_lookups.get("route_dep", {}),   0.0),
            ("AIRLINE_AVG_DEP_DELAY_PAST", historical_lookups.get("carrier_dep", {}), 0.0),
            ("ORIGIN_AVG_DEP_DELAY_PAST",  historical_lookups.get("origin_dep", {}),  0.0)
        ]:
            key_series = df_batch["ROUTE"] if "ROUTE" in col_name else (df_batch["OP_CARRIER"] if "AIRLINE" in col_name else df_batch["ORIGIN"])
            df_batch[col_name] = key_series.map(lookup_dict).fillna(default_val).astype("float32")
    
    # ═══════════════════════════════════════════════════════════════
    # 9. MỚI: Rolling 12-month delay averages
    # ═══════════════════════════════════════════════════════════════
    if rolling_lookups is not None:
        for col_name, lookup_dict, default_val in [
            ("ROUTE_AVG_ARR_DELAY_ROLLING",   rolling_lookups.get("route_arr", {}),   0.0),
            ("AIRLINE_AVG_ARR_DELAY_ROLLING", rolling_lookups.get("carrier_arr", {}), 0.0),
            ("ORIGIN_AVG_ARR_DELAY_ROLLING",  rolling_lookups.get("origin_arr", {}),  0.0),
            ("ROUTE_AVG_DEP_DELAY_ROLLING",   rolling_lookups.get("route_dep", {}),   0.0),
            ("AIRLINE_AVG_DEP_DELAY_ROLLING", rolling_lookups.get("carrier_dep", {}), 0.0),
            ("ORIGIN_AVG_DEP_DELAY_ROLLING",  rolling_lookups.get("origin_dep", {}),  0.0)
        ]:
            key_series = df_batch["ROUTE"] if "ROUTE" in col_name else (df_batch["OP_CARRIER"] if "AIRLINE" in col_name else df_batch["ORIGIN"])
            df_batch[col_name] = key_series.map(lookup_dict).fillna(default_val).astype("float32")
    
    # ═══════════════════════════════════════════════════════════════
    # 10. MỚI: Target Encoding - Tỷ lệ trễ lịch sử (Delay Rate)
    # ═══════════════════════════════════════════════════════════════
    if delay_rate_lookups is not None:
        for col_name, lookup_dict, default_val in [
            ("CARRIER_DELAY_RATE", delay_rate_lookups.get("carrier", {}), 0.0),
            ("ORIGIN_DELAY_RATE",  delay_rate_lookups.get("origin", {}),  0.0),
            ("ROUTE_DELAY_RATE",   delay_rate_lookups.get("route", {}),   0.0),
            ("DEST_DELAY_RATE",    delay_rate_lookups.get("dest", {}),    0.0)
        ]:
            key_series = (
                df_batch["ROUTE"] if "ROUTE" in col_name
                else df_batch["OP_CARRIER"] if "CARRIER" in col_name
                else df_batch["DEST"] if "DEST" in col_name
                else df_batch["ORIGIN"]
            )
            df_batch[col_name] = key_series.map(lookup_dict).fillna(default_val).astype("float32")
    
    # ═══════════════════════════════════════════════════════════════
    # 11. Các nhãn mục tiêu chuẩn
    # ═══════════════════════════════════════════════════════════════
    df_batch["IS_ARR_DELAY"] = (df_batch["ARR_DELAY"] >= 15).astype("int8")
    df_batch["IS_DEP_DELAY"] = (df_batch["DEP_DELAY"] >= 15).astype("int8")
    df_batch["ARR_DELAY"]    = df_batch["ARR_DELAY"].astype("float32")
    df_batch["DEP_DELAY"]    = df_batch["DEP_DELAY"].astype("float32")
    
    return df_batch

print("Hàm engineer_features_batch NÂNG CẤP đã sẵn sàng!")

Hàm engineer_features_batch NÂNG CẤP đã sẵn sàng!


### Cell 5: Quản lý Từ điển Categorical & Bộ tích lũy lịch sử (NÂNG CẤP)
Bao gồm 3 loại accumulator:
* **Cumulative**: Trung bình trễ tích lũy toàn bộ quá khứ (giống bản gốc)
* **Monthly**: Lưu trữ theo tháng để tính Rolling Window 12 tháng gần nhất
* **Delay Rate**: Tỷ lệ chuyến bay trễ ≥15 phút (cho Target Encoding)

In [6]:
# ═══════════════════════════════════════════════════════════════
# PHẦN 1: Từ điển mã hóa Categorical toàn cục
# ═══════════════════════════════════════════════════════════════
carrier_vocab = {}
airport_vocab = {}

def get_or_update_code(series, vocab):
    uniques = series.unique()
    for item in uniques:
        str_item = str(item)
        if str_item not in vocab:
            vocab[str_item] = len(vocab)
    return series.astype(str).map(vocab).astype("int16")

# ═══════════════════════════════════════════════════════════════
# PHẦN 2: Bộ tích lũy Cumulative (giống bản gốc)
# ═══════════════════════════════════════════════════════════════
historical_stats = {
    "carrier_arr": {}, "origin_arr": {}, "route_arr": {},
    "carrier_dep": {}, "origin_dep": {}, "route_dep": {}
}

def update_accumulators(df_year, accumulators):
    # Loại trừ năm Covid 2020 để tránh nhiễu
    if df_year["source_year"].iloc[0] == 2020:
        return accumulators
    
    for grp_col, stat_arr, stat_dep in [
        ("OP_CARRIER", "carrier_arr", "carrier_dep"),
        ("ORIGIN",     "origin_arr",  "origin_dep"),
        ("ROUTE",      "route_arr",   "route_dep")
    ]:
        agg_arr = df_year.groupby(grp_col)["ARR_DELAY"].agg(["sum", "count"]).to_dict("index")
        agg_dep = df_year.groupby(grp_col)["DEP_DELAY"].agg(["sum", "count"]).to_dict("index")
        
        for k, v in agg_arr.items():
            prev = accumulators[stat_arr].get(k, {"sum": 0.0, "count": 0})
            accumulators[stat_arr][k] = {"sum": prev["sum"] + v["sum"], "count": prev["count"] + v["count"]}
            
        for k, v in agg_dep.items():
            prev = accumulators[stat_dep].get(k, {"sum": 0.0, "count": 0})
            accumulators[stat_dep][k] = {"sum": prev["sum"] + v["sum"], "count": prev["count"] + v["count"]}
            
    return accumulators

def get_current_lookups(accumulators):
    lookups = {}
    for k, v in accumulators.items():
        lookups[k] = {grp: vals["sum"] / vals["count"] for grp, vals in v.items() if vals["count"] > 0}
    return lookups

# ═══════════════════════════════════════════════════════════════
# PHẦN 3: MỚI - Bộ tích lũy Monthly (cho Rolling Window)
# ═══════════════════════════════════════════════════════════════
historical_monthly_stats = {
    "carrier_arr": {}, "origin_arr": {}, "route_arr": {},
    "carrier_dep": {}, "origin_dep": {}, "route_dep": {}
}

def update_monthly_accumulators(df_year, monthly_accumulators, year):
    """Lưu trữ thống kê delay theo tháng để tính rolling window."""
    if year == 2020:  # Skip Covid
        return monthly_accumulators
    
    for grp_col, stat_arr, stat_dep in [
        ("OP_CARRIER", "carrier_arr", "carrier_dep"),
        ("ORIGIN",     "origin_arr",  "origin_dep"),
        ("ROUTE",      "route_arr",   "route_dep")
    ]:
        for month in df_year["MONTH"].unique():
            month_mask = df_year["MONTH"] == month
            month_data = df_year[month_mask]
            
            agg_arr = month_data.groupby(grp_col)["ARR_DELAY"].agg(["sum", "count"]).to_dict("index")
            agg_dep = month_data.groupby(grp_col)["DEP_DELAY"].agg(["sum", "count"]).to_dict("index")
            
            for k, v in agg_arr.items():
                if k not in monthly_accumulators[stat_arr]:
                    monthly_accumulators[stat_arr][k] = {}
                monthly_accumulators[stat_arr][k][(year, int(month))] = {"sum": float(v["sum"]), "count": int(v["count"])}
            
            for k, v in agg_dep.items():
                if k not in monthly_accumulators[stat_dep]:
                    monthly_accumulators[stat_dep][k] = {}
                monthly_accumulators[stat_dep][k][(year, int(month))] = {"sum": float(v["sum"]), "count": int(v["count"])}
    
    return monthly_accumulators

def get_rolling_lookups(monthly_accumulators, current_year, window_months=12):
    """Tính trung bình delay từ window_months tháng gần nhất trước current_year."""
    lookups = {}
    for stat_key, groups in monthly_accumulators.items():
        lookups[stat_key] = {}
        for grp_key, month_data in groups.items():
            total_sum = 0.0
            total_count = 0
            for (y, m), vals in month_data.items():
                if y < current_year:
                    months_ago = (current_year - y) * 12 + (1 - m)
                    if months_ago <= window_months:
                        total_sum += vals["sum"]
                        total_count += vals["count"]
            if total_count > 0:
                lookups[stat_key][grp_key] = total_sum / total_count
    return lookups

# ═══════════════════════════════════════════════════════════════
# PHẦN 4: MỚI - Bộ tích lũy Delay Rate (cho Target Encoding)
# ═══════════════════════════════════════════════════════════════
historical_delay_rate_stats = {
    "carrier": {},  # key -> {"delayed": count, "total": count}
    "origin": {},
    "route": {},
    "dest": {}
}

def update_delay_rate_accumulators(df_year, rate_accumulators):
    """Cập nhật tỷ lệ trễ ≥15 phút cho Target Encoding."""
    if df_year["source_year"].iloc[0] == 2020:
        return rate_accumulators
    for grp_col, stat_key in [
        ("OP_CARRIER", "carrier"),
        ("ORIGIN",     "origin"),
        ("ROUTE",      "route"),
        ("DEST",       "dest")
    ]:
        agg = df_year.groupby(grp_col)["IS_ARR_DELAY"].agg(["sum", "count"]).to_dict("index")
        for k, v in agg.items():
            prev = rate_accumulators[stat_key].get(k, {"delayed": 0, "total": 0})
            rate_accumulators[stat_key][k] = {
                "delayed": prev["delayed"] + int(v["sum"]),
                "total": prev["total"] + int(v["count"])
            }
    return rate_accumulators

def get_delay_rate_lookups(rate_accumulators):
    """Tính tỷ lệ trễ hiện tại từ accumulator."""
    lookups = {}
    for stat_key, groups in rate_accumulators.items():
        lookups[stat_key] = {}
        for grp_key, vals in groups.items():
            if vals["total"] > 0:
                lookups[stat_key][grp_key] = vals["delayed"] / vals["total"]
    return lookups

# ═══════════════════════════════════════════════════════════════
# PHẦN 5: Danh sách cột đặc trưng đầu ra NÂNG CẤP
# ═══════════════════════════════════════════════════════════════
features_common = [
    # === Đặc trưng lịch trình gốc (15 features) ===
    "DISTANCE_KM", "CRS_ELAPSED_TIME", "DEP_HOUR", "ARR_HOUR",
    "MONTH", "DAY", "DOW", "QUARTER", "IS_WEEKEND", "PEAK_HOUR",
    "SEASON_CODE", "TIME_OF_DAY_CODE", "DISTANCE_GROUP_CODE", "LONG_HAUL",
    "IS_COVID_PERIOD",
    # === Categorical codes (3 features) ===
    "OP_CARRIER_CODE", "ORIGIN_CODE", "DEST_CODE",
    # === MỚI: Thời tiết (17 features) ===
    "O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD",
    "IS_ORIGIN_RAINY", "IS_DEST_RAINY", "IS_ORIGIN_WINDY", "IS_DEST_WINDY",
    "IS_ORIGIN_COLD", "IS_DEST_COLD",
    "TEMP_DIFF", "WSPD_DIFF",
    "WEATHER_SEVERITY_ORIGIN", "WEATHER_SEVERITY_DEST", "WEATHER_SEVERITY_TOTAL",
    # === MỚI: Cyclical encoding (6 features) ===
    "MONTH_SIN", "MONTH_COS", "HOUR_SIN", "HOUR_COS", "DOW_SIN", "DOW_COS",
    # === MỚI: Congestion nâng cao (4 features) ===
    "ORIGIN_CONGESTION_SAME_HOUR", "DEST_CONGESTION_SAME_HOUR",
    "ORIGIN_DAILY_FLIGHTS", "DEST_DAILY_FLIGHTS",
    # === MỚI: Target encoding - Delay rate (4 features) ===
    "CARRIER_DELAY_RATE", "ORIGIN_DELAY_RATE", "ROUTE_DELAY_RATE", "DEST_DELAY_RATE"
]

features_arr = features_common + [
    "ROUTE_AVG_ARR_DELAY_PAST", "AIRLINE_AVG_ARR_DELAY_PAST", "ORIGIN_AVG_ARR_DELAY_PAST",
    "ROUTE_AVG_ARR_DELAY_ROLLING", "AIRLINE_AVG_ARR_DELAY_ROLLING", "ORIGIN_AVG_ARR_DELAY_ROLLING"
]

features_dep = features_common + [
    "ROUTE_AVG_DEP_DELAY_PAST", "AIRLINE_AVG_DEP_DELAY_PAST", "ORIGIN_AVG_DEP_DELAY_PAST",
    "ROUTE_AVG_DEP_DELAY_ROLLING", "AIRLINE_AVG_DEP_DELAY_ROLLING", "ORIGIN_AVG_DEP_DELAY_ROLLING"
]

# Khởi tạo thư mục đích lưu trữ
split_base_dirs = [
    repo_root / "data" / "split",
    repo_root / "src" / "data" / "split"
]

tasks = [
    "arrival_classification", "departure_classification",
    "arrival_regression",     "departure_regression"
]

for base in split_base_dirs:
    for task in tasks:
        for fold in ["train", "valid", "test"]:
            (base / task / fold).mkdir(parents=True, exist_ok=True)

print(f"Tổng số features chung: {len(features_common)}")
print(f"Tổng số features arrival: {len(features_arr)}")
print(f"Tổng số features departure: {len(features_dep)}")
print("Tất cả thư mục split đã sẵn sàng!")

Tổng số features chung: 49
Tổng số features arrival: 55
Tổng số features departure: 55
Tất cả thư mục split đã sẵn sàng!


### Cell 6: BỘ ĐIỀU PHỐI CHÍNH - Vòng lặp Xử lý theo từng năm (NÂNG CẤP)
Quy trình thực hiện tuần tự qua từng năm:
1. Đọc dữ liệu của 1 năm (`year=YYYY`) với **23 cột** (thêm 6 cột thời tiết).
2. Làm sạch và kiểm toán logic.
3. Mã hóa Categorical toàn cục.
4. Tính lookups từ 3 loại accumulator (cumulative, rolling, delay rate).
5. Tạo lập toàn bộ đặc trưng NÂNG CẤP.
6. Cập nhật 3 loại accumulator.
7. Xuất **X** và **y** Parquet riêng biệt cho 4 bài toán.
8. Giải phóng RAM triệt để (`del df; gc.collect()`).

In [7]:
execution_summary = []

print("=" * 80)
print(f"BẮT ĐẦU VÒNG LẶP XỬ LÝ BATCHING THEO NĂM: {YEARS_TO_PROCESS}")
print(f"NGUỒN DỮ LIỆU: {DATA_SOURCE}")
print(f"TỔNG SỐ FEATURES: {len(features_arr)} (arrival) / {len(features_dep)} (departure)")
print("=" * 80)

total_start_time = time.time()

for year in YEARS_TO_PROCESS:
    year_start = time.time()
    year_dir = data_dir / f"year={year}"
    
    if not year_dir.exists():
        print(f"-> Bỏ qua năm {year}: Không tìm thấy thư mục {year_dir}")
        continue
        
    print(f"\n--- [NĂM {year}] Đang nạp và xử lý... ---")
    
    # 1. Đọc 23 cột từ đĩa (bao gồm thời tiết)
    df_year = pd.read_parquet(year_dir, columns=PROJECTION_COLUMNS)
    raw_rows = len(df_year)
    
    # 2. Làm sạch dữ liệu
    df_year, removed_rows = clean_and_filter_batch(df_year)
    clean_rows = len(df_year)
    
    # 3. Mã hóa từ điển Categorical toàn cục
    df_year["OP_CARRIER_CODE"] = get_or_update_code(df_year["OP_CARRIER"], carrier_vocab)
    df_year["ORIGIN_CODE"]     = get_or_update_code(df_year["ORIGIN"], airport_vocab)
    df_year["DEST_CODE"]       = get_or_update_code(df_year["DEST"], airport_vocab)
    
    # 4. Tính lookups từ 3 loại accumulator
    current_lookups = get_current_lookups(historical_stats)
    rolling_lookups = get_rolling_lookups(historical_monthly_stats, year)
    delay_rate_lookups = get_delay_rate_lookups(historical_delay_rate_stats)
    
    # 5. Tạo lập toàn bộ đặc trưng NÂNG CẤP
    df_year = engineer_features_batch(
        df_year,
        historical_lookups=current_lookups,
        rolling_lookups=rolling_lookups,
        delay_rate_lookups=delay_rate_lookups
    )
    
    # 6. Cập nhật 3 loại accumulator cho các năm tương lai
    historical_stats = update_accumulators(df_year, historical_stats)
    historical_monthly_stats = update_monthly_accumulators(df_year, historical_monthly_stats, year)
    historical_delay_rate_stats = update_delay_rate_accumulators(df_year, historical_delay_rate_stats)
    
    # 7. Xác định fold lưu trữ
    if year <= 2022:
        fold_name = "train"
    elif year == 2023:
        fold_name = "valid"
    else:
        fold_name = "test"
    
    # 8. Xuất X và y Parquet RIÊNG BIỆT (tương thích tabular_classification_notebook)
    part_filename = f"part_{year}.parquet"
    
    X_arr = df_year[features_arr].copy()
    X_dep = df_year[features_dep].copy()
    
    y_arr_cls = df_year[["IS_ARR_DELAY"]].rename(columns={"IS_ARR_DELAY": "target"}).copy()
    y_dep_cls = df_year[["IS_DEP_DELAY"]].rename(columns={"IS_DEP_DELAY": "target"}).copy()
    y_arr_reg = df_year[["ARR_DELAY"]].rename(columns={"ARR_DELAY": "target"}).copy()
    y_dep_reg = df_year[["DEP_DELAY"]].rename(columns={"DEP_DELAY": "target"}).copy()
    
    for base in split_base_dirs:
        # Arrival Classification
        X_arr.to_parquet(base / "arrival_classification" / fold_name / f"X_{part_filename}", index=False)
        y_arr_cls.to_parquet(base / "arrival_classification" / fold_name / f"y_{part_filename}", index=False)
        
        # Departure Classification
        X_dep.to_parquet(base / "departure_classification" / fold_name / f"X_{part_filename}", index=False)
        y_dep_cls.to_parquet(base / "departure_classification" / fold_name / f"y_{part_filename}", index=False)
        
        # Arrival Regression
        X_arr.to_parquet(base / "arrival_regression" / fold_name / f"X_{part_filename}", index=False)
        y_arr_reg.to_parquet(base / "arrival_regression" / fold_name / f"y_{part_filename}", index=False)
        
        # Departure Regression
        X_dep.to_parquet(base / "departure_regression" / fold_name / f"X_{part_filename}", index=False)
        y_dep_reg.to_parquet(base / "departure_regression" / fold_name / f"y_{part_filename}", index=False)
    
    # Đo lường hiệu năng và RAM
    elapsed = time.time() - year_start
    ram_mb = get_memory_usage_mb()
    
    execution_summary.append({
        "Năm": year,
        "Fold": fold_name.upper(),
        "Số dòng gốc": f"{raw_rows:,}",
        "Số dòng sạch": f"{clean_rows:,}",
        "Tỷ lệ trễ đến": f"{y_arr_cls['target'].mean()*100:.2f}%",
        "Tỷ lệ trễ cất cánh": f"{y_dep_cls['target'].mean()*100:.2f}%",
        "Thời gian chạy": f"{elapsed:.1f}s",
        "RAM đỉnh": f"{ram_mb:.1f} MB"
    })
    
    print(f"-> [Hoàn thành năm {year}]: {clean_rows:,} dòng sạch ({fold_name.upper()}) | Chạy trong: {elapsed:.1f}s | RAM hiện tại: {ram_mb:.1f} MB")
    
    # 9. GIẢI PHÓNG BỘ NHỚ TRIỆT ĐỂ CHO BATCH TIẾP THEO
    del df_year, X_arr, X_dep, y_arr_cls, y_dep_cls, y_arr_reg, y_dep_reg
    gc.collect()

total_time = time.time() - total_start_time
print("\n" + "=" * 80)
print(f"TOÀN BỘ 9 NĂM ĐÃ HOÀN TẤT THÀNH CÔNG TRONG {total_time/60:.2f} PHÚT!")
print("=" * 80)

BẮT ĐẦU VÒNG LẶP XỬ LÝ BATCHING THEO NĂM: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
NGUỒN DỮ LIỆU: tabular_by_year
TỔNG SỐ FEATURES: 55 (arrival) / 55 (departure)

--- [NĂM 2016] Đang nạp và xử lý... ---
-> [Hoàn thành năm 2016]: 5,537,985 dòng sạch (TRAIN) | Chạy trong: 83.3s | RAM hiện tại: 6433.7 MB

--- [NĂM 2017] Đang nạp và xử lý... ---
-> [Hoàn thành năm 2017]: 5,575,871 dòng sạch (TRAIN) | Chạy trong: 102.2s | RAM hiện tại: 6541.2 MB

--- [NĂM 2018] Đang nạp và xử lý... ---
-> [Hoàn thành năm 2018]: 6,986,835 dòng sạch (TRAIN) | Chạy trong: 148.5s | RAM hiện tại: 8147.6 MB

--- [NĂM 2019] Đang nạp và xử lý... ---
-> [Hoàn thành năm 2019]: 7,161,819 dòng sạch (TRAIN) | Chạy trong: 149.1s | RAM hiện tại: 8327.6 MB

--- [NĂM 2020] Đang nạp và xử lý... ---
-> [Hoàn thành năm 2020]: 4,312,072 dòng sạch (TRAIN) | Chạy trong: 71.6s | RAM hiện tại: 5208.9 MB

--- [NĂM 2021] Đang nạp và xử lý... ---
-> [Hoàn thành năm 2021]: 5,755,632 dòng sạch (TRAIN) | Chạy trong: 107.5s 

### Cell 7: Báo cáo Tổng kết & Kiểm định Toàn diện Dữ liệu Đầu ra (Final Report & Verification)

In [8]:
df_summary = pd.DataFrame(execution_summary)
print("=" * 90)
print("BẢNG TỔNG HỢP TIẾN ĐỘ & KIỂM TOÁN DỮ LIỆU TỪNG NĂM")
print("=" * 90)
print(df_summary.to_string(index=False))

print("\n" + "=" * 90)
print("THỐNG KÊ FEATURES NÂNG CẤP:")
print("=" * 90)
print(f"Tổng features chung (common):     {len(features_common)}")
print(f"Tổng features arrival:            {len(features_arr)}")
print(f"Tổng features departure:          {len(features_dep)}")
print(f"Trong đó features MỚI:")
print(f"  - Thời tiết:                    17 features")
print(f"  - Cyclical encoding:             6 features")
print(f"  - Congestion nâng cao:           4 features (gồm 1 cũ + 3 mới)")
print(f"  - Target encoding (delay rate):  4 features")
print(f"  - Rolling window delays:         6 features (per task)")

print("\n" + "=" * 90)
print("KIỂM ĐỊNH TÍNH SẴN SÀNG CỦA CÁC THƯ MỤC SPLIT:")
print("=" * 90)

for task in tasks:
    print(f"\n--- THƯ MỤC NHIỆM VỤ: {task} ---")
    for fold in ["train", "valid", "test"]:
        fold_dir = repo_root / "data" / "split" / task / fold
        x_files = list(fold_dir.glob("X_*.parquet"))
        y_files = list(fold_dir.glob("y_*.parquet"))
        print(f"   * [{fold.upper():5s}]: {len(x_files)} tệp đặc trưng X, {len(y_files)} tệp nhãn y ({fold_dir})")

print("\n" + "=" * 90)
print("HOÀN TẤT! Notebook huấn luyện có thể nạp dữ liệu với ~50 features mới.")
print("=" * 90)

BẢNG TỔNG HỢP TIẾN ĐỘ & KIỂM TOÁN DỮ LIỆU TỪNG NĂM
 Năm  Fold Số dòng gốc Số dòng sạch Tỷ lệ trễ đến Tỷ lệ trễ cất cánh Thời gian chạy  RAM đỉnh
2016 TRAIN   5,537,987    5,537,985        17.41%             17.12%          83.3s 6433.7 MB
2017 TRAIN   5,575,872    5,575,871        18.45%             18.08%         102.2s 6541.2 MB
2018 TRAIN   6,986,842    6,986,835        19.09%             18.33%         148.5s 8147.6 MB
2019 TRAIN   7,161,827    7,161,819        19.11%             18.63%         149.1s 8327.6 MB
2020 TRAIN   4,312,091    4,312,072         9.73%              9.00%          71.6s 5208.9 MB
2021 TRAIN   5,755,666    5,755,632        17.08%             17.27%         107.5s 6853.4 MB
2022 TRAIN   6,413,416    6,413,360        20.99%             21.25%         104.3s 7595.7 MB
2023 VALID   6,645,461    6,645,342        20.54%             20.47%         107.6s 7885.5 MB
2024  TEST   6,284,841    6,284,739        20.81%             20.58%         105.8s 7564.0 MB

THỐNG KÊ